In [1]:
# !pip uninstall -y cudf-cu12 pylibcudf-cu12
# !pip install -U transformers datasets accelerate -q


In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
  DistilBertTokenizerFast,
  DistilBertForSequenceClassification,
  Trainer,
  TrainingArguments,
  pipeline
)
from datasets import Dataset

In [3]:
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [4]:
# Load and preprocess training data

train_df = pd.read_csv("First999Labels_with_clean.csv")


# Keep only rows 2 to 1001 (Python indexing: 1:1001)
train_df = train_df.iloc[1:1001]


# Clean text and map labels
train_df = train_df[["clean", "majority_vote"]].dropna(subset=["clean", "majority_vote"])


# Map labels: 1 -> 0, 2 -> 1, 3 -> 2
label_map = {1: 0, 2: 1, 3: 2}
train_df["label"] = train_df["majority_vote"].map(label_map)


# Split into train/validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
train_df["clean"].tolist(),
train_df["label"].tolist(),
test_size=0.1,
random_state=42
)

In [5]:
# tokenization

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")


train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)


train_dataset = Dataset.from_dict({**train_encodings, "labels": train_labels})
val_dataset = Dataset.from_dict({**val_encodings, "labels": val_labels})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# model setup

model = DistilBertForSequenceClassification.from_pretrained(
"distilbert-base-uncased", num_labels=3
)


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)


trainer = Trainer(
model=model,
args=training_args,
train_dataset=train_dataset,
eval_dataset=val_dataset,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# train and save model

trainer.train()

# Save model and tokenizer
save_path = "distilbert_sentiment_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


print(f"Model saved to {save_path}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ethanlu0929 (ethanlu0929-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,1.005300
20,0.888200
30,0.889200
40,0.524100
50,0.803400
60,0.766900
70,0.850900
80,0.710100
90,0.650600
100,0.775300


Model saved to distilbert_sentiment_model


In [15]:
# # apply distilbert model to rest of combined_forum_cleaned dataset

# combined_df = pd.read_csv("combined_forum_cleaned.csv")


# # Select unseen data (after first 1000 entries)
# unseen_df = combined_df.iloc[1000:].copy()


# # Replace blank or missing comments with '.'
# unseen_df["stopword_clean"] = unseen_df["stopword_clean"].fillna("").apply(lambda x: x.strip() if isinstance(x, str) else "")
# unseen_df.loc[unseen_df["stopword_clean"] == "", "stopword_clean"] = "."


# # Load trained model for inference
# model_loaded = DistilBertForSequenceClassification.from_pretrained(save_path)
# tokenizer_loaded = DistilBertTokenizerFast.from_pretrained(save_path)


# # Build sentiment analysis pipeline
# sentiment_analyzer = pipeline(
# "text-classification",
# model=model_loaded,
# tokenizer=tokenizer_loaded,
# truncation=True,
# padding=True,
# device=0 if torch.cuda.is_available() else -1
# )


# print("Running sentiment analysis on remaining comments...")
# preds = sentiment_analyzer(list(unseen_df["stopword_clean"]), batch_size=16)


# unseen_df["predicted_label"] = [p["label"] for p in preds]
# unseen_df["predicted_score"] = [p["score"] for p in preds]

# --- Apply DistilBERT model to rest of combined_forum_cleaned dataset ---


# Load dataset
combined_df = pd.read_csv("combined_forum_cleaned.csv")

# Select unseen data (after first 1000 entries)
unseen_df = combined_df.iloc[1000:].copy()

# Replace blank or missing comments with '.'
unseen_df["stopword_clean"] = unseen_df["stopword_clean"].fillna("").apply(lambda x: x.strip() if isinstance(x, str) else "")
unseen_df.loc[unseen_df["stopword_clean"] == "", "stopword_clean"] = "."

# Load trained model and tokenizer for inference
model_loaded = DistilBertForSequenceClassification.from_pretrained(save_path)
tokenizer_loaded = DistilBertTokenizerFast.from_pretrained(save_path)

# Build sentiment analysis pipeline
sentiment_analyzer = pipeline(
    "text-classification",
    model=model_loaded,
    tokenizer=tokenizer_loaded,
    truncation=True,
    padding=True,
    device=0 if torch.cuda.is_available() else -1
)

print("Running sentiment analysis on remaining comments...")
preds = sentiment_analyzer(list(unseen_df["stopword_clean"]), batch_size=16)

# Extract predictions
unseen_df["predicted_label"] = [p["label"] for p in preds]
unseen_df["predicted_score"] = [p["score"] for p in preds]

# --- Map LABEL_X to human-readable sentiment ---
label_text_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
unseen_df["predicted_label"] = unseen_df["predicted_label"].map(label_text_map)

# --- Merge stopword_clean and clean (if both exist) ---
if "stopword_clean" in unseen_df.columns:
    if "clean" in unseen_df.columns:
        unseen_df["clean"] = unseen_df["clean"].fillna("") + " " + unseen_df["stopword_clean"].fillna("")
    else:
        unseen_df["clean"] = unseen_df["stopword_clean"]
    unseen_df = unseen_df.drop(columns=["stopword_clean"], errors="ignore")

# --- Drop unused columns ---
columns_to_drop = ["body", "ID", "course ID"]
unseen_df = unseen_df.drop(columns=[c for c in columns_to_drop if c in unseen_df.columns], errors="ignore")


Device set to use cuda:0


Running sentiment analysis on remaining comments...


In [16]:
# combine the results

# Copy the training subset
first_1000 = train_df.copy()

# Rename label column to predicted_label for consistency
first_1000 = first_1000.rename(columns={"label": "predicted_label"})

# Convert numeric labels back to sentiment text
numeric_label_map = {0: "negative", 1: "neutral", 2: "positive"}
first_1000["predicted_label"] = first_1000["predicted_label"].map(numeric_label_map)

# Merge stopword_clean + clean for first_1000 as well
if "stopword_clean" in first_1000.columns:
    if "clean" in first_1000.columns:
        first_1000["clean"] = first_1000["clean"].fillna("") + " " + first_1000["stopword_clean"].fillna("")
    else:
        first_1000["clean"] = first_1000["stopword_clean"]
    first_1000 = first_1000.drop(columns=["stopword_clean"], errors="ignore")

# Drop unused columns
columns_to_drop = ["body", "ID", "course ID"]
first_1000 = first_1000.drop(columns=[c for c in columns_to_drop if c in first_1000.columns], errors="ignore")

# --- Combine both parts ---
final_df = pd.concat([first_1000, unseen_df], ignore_index=True)

# --- Export to CSV ---
final_df.to_csv("combined_forum_with_predictions.csv", index=False)

print("All predictions saved to combined_forum_with_predictions.csv")
print(final_df.head())

All predictions saved to combined_forum_with_predictions.csv
                                               clean  majority_vote  \
0                                          smartbook            2.0   
1  answer question tried everything nothing seems...            1.0   
2  passed output printed bubble line misleading u...            1.0   
3  problem question try advise bogus variable ans...            2.0   
4                                         around ord            2.0   

  predicted_label   id course_id  predicted_score  
0         neutral  NaN       NaN              NaN  
1        negative  NaN       NaN              NaN  
2        negative  NaN       NaN              NaN  
3         neutral  NaN       NaN              NaN  
4         neutral  NaN       NaN              NaN  
